# 三一重工（600031.SH）— 每日交易数据获取与可视化

本 Notebook 演示如何从 **Tushare Pro** 获取 A 股日线数据并进行可视化分析。

**数据范围**：过去一年（2025-07-02 ~ 2026-07-02）

---

## 1. 环境准备与数据获取

> 注：默认读取本地 CSV 缓存。如需从 Tushare API 实时拉取，将下方 `USE_CACHED` 改为 `False` 并填入你的 Token。

In [ ]:
import os
import pandas as pd
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# 中文显示设置
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ---- 配置 ----
TS_CODE = '600031.SH'
CSV_PATH = '/Users/xuyajing/Desktop/AI/Quant/sany_600031_daily.csv'
USE_CACHED = True   # True=读本地CSV, False=从API拉取

if USE_CACHED:
    if os.path.exists(CSV_PATH):
        df = pd.read_csv(CSV_PATH)
        df['trade_date'] = pd.to_datetime(df['trade_date'])
        df.sort_values('trade_date', inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f'✅ 从本地 CSV 读取 {len(df)} 条记录')
    else:
        print('⚠️ 找不到本地 CSV，请将 USE_CACHED 设为 False 从 Tushare API 拉取')
else:
    import tushare as ts
    TOKEN = '你的 Tushare Token'
    pro = ts.pro_api(TOKEN)
    df_raw = pro.daily(ts_code=TS_CODE, start_date='20250702', end_date='20260702')
    df = df_raw.sort_values('trade_date').reset_index(drop=True)
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    print(f'✅ 从 Tushare API 获取 {len(df)} 条记录')

print('✅ 环境准备完成')
df.head()

## 2. 数据概览

In [ ]:
print('--- 数据基本信息 ---')
print(f'日期范围: {df["trade_date"].min().date()} ~ {df["trade_date"].max().date()}')
print(f'交易日数: {len(df)}')
print(f'字段: {list(df.columns)}\n')

print('--- 描述性统计 ---')
df[['open', 'high', 'low', 'close', 'vol', 'pct_chg']].describe().round(2)

In [ ]:
first_close = df['close'].iloc[0]
last_close = df['close'].iloc[-1]
total_return = (last_close - first_close) / first_close * 100
max_close = df['close'].max()
min_close = df['close'].min()
avg_close = df['close'].mean()

print(f'📊 关键指标')
print(f'{"━"*38}')
print(f'起始收盘价: ¥{first_close:.2f}')
print(f'最新收盘价: ¥{last_close:.2f}')
print(f'期间涨跌幅: {total_return:+.2f}%')
print(f'最高收盘价: ¥{max_close:.2f}')
print(f'最低收盘价: ¥{min_close:.2f}')
print(f'均价:     ¥{avg_close:.2f}')
print(f'{"━"*38}')

## 3. 收盘价走势曲线

> A 股约定：涨 = 🔴 红色，跌 = 🟢 绿色

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5.5))

ax.plot(df['trade_date'], df['close'], color='#e74c3c', linewidth=1.5, label='收盘价')
ax.fill_between(df['trade_date'], df['close'], df['close'].min(), alpha=0.06, color='#e74c3c')

# 均价线
avg = df['close'].mean()
ax.axhline(y=avg, color='#5b8ff9', linestyle='--', linewidth=1, alpha=0.6)
ax.text(df['trade_date'].iloc[-1], avg, f' 均价 ¥{avg:.2f}', va='bottom', fontsize=10, color='#5b8ff9')

# 标注最高最低
max_idx, min_idx = df['close'].idxmax(), df['close'].idxmin()
ax.annotate(f'¥{df["close"].max():.2f}', xy=(df['trade_date'].iloc[max_idx], df['close'].max()),
            xytext=(15,15), textcoords='offset points', arrowprops=dict(arrowstyle='->', color='#e74c3c'), fontsize=10, color='#e74c3c')
ax.annotate(f'¥{df["close"].min():.2f}', xy=(df['trade_date'].iloc[min_idx], df['close'].min()),
            xytext=(15,-20), textcoords='offset points', arrowprops=dict(arrowstyle='->', color='#27ae60'), fontsize=10, color='#27ae60')

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.set_title('三一重工 (600031.SH) 每日收盘价走势', fontsize=15, fontweight='bold')
ax.set_xlabel('交易日期', fontsize=11)
ax.set_ylabel('收盘价 (¥)', fontsize=11)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.25, linestyle='--')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
plt.close('all')

## 4. 收盘价 + 成交量双面板

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

# 上：收盘价
ax1.plot(df['trade_date'], df['close'], color='#e74c3c', linewidth=1.5)
ax1.axhline(y=avg, color='#5b8ff9', linestyle='--', linewidth=1, alpha=0.6)
ax1.set_ylabel('收盘价 (¥)', fontsize=11)
ax1.set_title('三一重工 (600031.SH) — 收盘价与成交量', fontsize=15, fontweight='bold')
ax1.grid(True, alpha=0.25, linestyle='--')
ax1.legend(['收盘价', f'均价 ¥{avg:.2f}'], loc='upper left')

# 下：成交量
colors_vol = ['#e74c3c' if df['pct_chg'].iloc[i] >= 0 else '#27ae60' for i in range(len(df))]
ax2.bar(df['trade_date'], df['vol'] / 10000, color=colors_vol, alpha=0.65, width=0.8)
ax2.set_xlabel('交易日期', fontsize=11)
ax2.set_ylabel('成交量 (万股)', fontsize=11)
ax2.grid(True, alpha=0.25, linestyle='--')

plt.tight_layout()
plt.show()
plt.close('all')

## 5. 涨跌幅分布

In [ ]:
up_days = (df['pct_chg'] > 0).sum()
down_days = (df['pct_chg'] < 0).sum()
flat_days = (df['pct_chg'] == 0).sum()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.hist(df['pct_chg'], bins=40, color='#5b8ff9', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_xlabel('涨跌幅 (%)', fontsize=11)
ax.set_ylabel('交易日数', fontsize=11)
ax.set_title('三一重工 — 每日涨跌幅分布', fontsize=14, fontweight='bold')
ax.text(0.97, 0.95, f'涨 {up_days} 天 | 跌 {down_days} 天 | 平 {flat_days} 天',
        transform=ax.transAxes, ha='right', va='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.grid(True, alpha=0.25, linestyle='--')
plt.tight_layout()
plt.show()
plt.close('all')

## 6. 移动平均线（MA10 / MA20）

In [ ]:
df['MA10'] = df['close'].rolling(window=10).mean()
df['MA20'] = df['close'].rolling(window=20).mean()

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.plot(df['trade_date'], df['close'], color='#e74c3c', linewidth=1.3, alpha=0.7, label='收盘价')
ax.plot(df['trade_date'], df['MA10'], color='#f39c12', linewidth=1.5, label='MA10')
ax.plot(df['trade_date'], df['MA20'], color='#5b8ff9', linewidth=1.5, label='MA20')
ax.set_title('三一重工 (600031.SH) — 移动平均线', fontsize=15, fontweight='bold')
ax.set_xlabel('交易日期', fontsize=11)
ax.set_ylabel('价格 (¥)', fontsize=11)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.25, linestyle='--')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()
plt.close('all')

---
**数据来源**：[Tushare Pro](https://tushare.pro)  |  **声明**：本分析仅供参考，不构成投资建议